## Separate Columns

In [0]:
df_reseller_clean = spark.read.csv("/Volumes/final_project/bronze/data/Reseller.csv" , header=True, sep="\t")

display(df_reseller_clean)

In [0]:
df_targets_clean = spark.read.csv("/Volumes/final_project/bronze/data/Targets.csv" , header=True, sep="\t")

display(df_targets_clean)

## Change Data Types

In [0]:
#From Reseller change ResellerKey to int
from pyspark.sql.functions import col

df_reseller_clean = df_reseller_clean.withColumn("ResellerKey", col("ResellerKey").cast("int"))

df_reseller_clean.printSchema()

In [0]:
from pyspark.sql.functions import col, regexp_replace

# 1. Remove $ from targets and change data type to decimal(10,2)
df_targets_clean = df_targets_clean.withColumn(
    "Target", 
    (regexp_replace(col("Target"), r"[\$,]", "").cast("int"))
)

# 2. Change EmployeeID to int
df_targets_clean = df_targets_clean.withColumn("EmployeeID", col("EmployeeID").cast("int"))

# Display result
display(df_targets_clean)

df_targets_clean.printSchema()

## Change value format from Reseller , where UK to United Kingdom

In [0]:
#Change Format from Reseller , where UK to United Kingdom
from pyspark.sql.functions import regexp_replace

df_reseller_clean = df_reseller_clean.withColumn("Country-Region", regexp_replace(col("Country-Region"), "UK", "United Kingdom"))

display(df_reseller_clean)

## Change culumn titles format as snake_tale

In [0]:
#snake_case to Reseller
# 1. Change all columns to snake_case
new_cols = [c.replace(" ", "_").replace("-", "_").lower() for c in df_reseller_clean.columns]
df_reseller_clean = df_reseller_clean.toDF(*new_cols)

# 2. Rename resellerkey to reseller_key
df_reseller_clean = df_reseller_clean.withColumnRenamed("resellerkey", "reseller_key")

display(df_reseller_clean)

In [0]:
#snake_case to Targets
# 1. Change all columns to snake_case
new_cols = [c.replace(" ", "_").replace("-", "_").lower() for c in df_targets_clean.columns]
df_targets_clean = df_targets_clean.toDF(*new_cols)

# 2. Rename EmployeeID to employee_id
df_targets_clean = df_targets_clean.withColumnRenamed("employeeid", "employee_id")

# 3. Rename TargetMonth to target_mont
df_targets_clean = df_targets_clean.withColumnRenamed("targetmonth" , "target_date")

display(df_targets_clean)

## Change string type to date type from Targets

In [0]:
# Fix the Spark policy
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

from pyspark.sql import functions as F

# Apply the conversion (ensure you use the correct column name from your df)
df_targets_clean = df_targets_clean.withColumn(
    "target_date", 
    F.to_date(F.trim(F.col("target_date")), "EEEE, MMMM d, yyyy")
)

display(df_targets_clean)


## Check for Duplicates and remove them

In [0]:
#check for duplicates in reseller
total_count = df_reseller_clean.count()
unique_count = df_reseller_clean.distinct().count()

print(f"Total Rows: {total_count}")
print(f"Unique Rows: {unique_count}")
print(f"Duplicate Rows found: {total_count - unique_count}")

In [0]:
#check for duplicates in targets
total_count = df_targets_clean.count()
unique_count = df_targets_clean.distinct().count()

print(f"Total Rows: {total_count}")
print(f"Unique Rows: {unique_count}")
print(f"Duplicate Rows found: {total_count - unique_count}")


In [0]:
#WE DON'T HAVE DUPLICATE ROWS IN RESELLER AND TARGET

## Check for nulls

In [0]:
#check for nulls in reseller
from pyspark.sql import functions as F

# Create a count of nulls for every column
null_counts_reseller = df_reseller_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_reseller_clean.columns
])

display(null_counts_reseller)

In [0]:
#check for nulls in targets
from pyspark.sql import functions as F

# Create a count of nulls for every column
null_counts_targets = df_targets_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_targets_clean.columns
])

display(null_counts_targets)

In [0]:
#WE DON'T HAVE NULL VALUES IN RESELLER AND TARGET

## Create Delta Tables and store them in Silver layer

In [0]:
# 1. Ensure the schema exists in that specific catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS final_project.silver")

# 2. Write the table using the full 3-part name
df_reseller_clean.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("final_project.silver.reseller")

# 3. Verify the data
display(spark.table("final_project.silver.reseller"))

In [0]:
# 1. Ensure the schema exists in that specific catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS final_project.silver")

# 2. Write the table using the full 3-part name
df_targets_clean.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("final_project.silver.targets")

# 3. Verify the data
display(spark.table("final_project.silver.targets"))

In [0]:
#if we want to drop the table targets from the silver schema of final_project catalog
#spark.sql("DROP TABLE IF EXISTS final_project.silver.targets")